In [1]:
import pandas as pd
import pdfplumber
import re

In [2]:
PDF_PATH = 'Queensland-Government-Gazette.pdf'

pdf = pdfplumber.open(PDF_PATH)
print(f'Total pages: {len(pdf.pages)}')

Total pages: 58


In [3]:
# Schedule 3 (WIC table) runs from page 31 to 58.
# Each data row: WIC_CODE Description RATE
# Some descriptions wrap across two lines — the orphan description
# precedes the WIC code line, which then has only code + rate.
# Group Training Organisation entries (8101G1-G4) list sub-WIC ranges
# on continuation lines — skip those.
# Labour hire divisional WICs (A01100, B06000, etc.) use alpha prefix.

WIC_TABLE_START = 30  # 0-indexed page 31
WIC_TABLE_END = 58    # through page 58

all_lines = []
for page in pdf.pages[WIC_TABLE_START:WIC_TABLE_END]:
    text = page.extract_text()
    if text:
        for line in text.split('\n'):
            all_lines.append(line.strip())

print(f'Total lines extracted from Schedule 3: {len(all_lines)}')

Total lines extracted from Schedule 3: 1028


In [4]:
# Parse WIC data rows.
# Full row: CODE DESCRIPTION RATE  (e.g. "011103 Nursery Production (Under Cover) 2.759")
# Wrapped rows: 3 descriptions are too long and wrap across 3 lines:
#   line N:   "Sheet Metal Product Manufacturing (except Metal Structural and"
#   line N+1: "224019 3.377"   (code + rate only)
#   line N+2: "Container Products)"   (tail of description)
# Labour hire WICs use alpha prefix (e.g. A01100).

full_pattern = re.compile(r'^(\d{6}|8101G\d|[A-S]\d{5})\s+(.+?)\s+(\d+\.\d+)\s*$')
short_pattern = re.compile(r'^(\d{6}|8101G\d|[A-S]\d{5})\s+(\d+\.\d+)\s*$')

# Lines that are sub-WIC listings under Group Training entries (ranges like "014264 - 014409, ...")
range_pattern = re.compile(r'^\d{6}\s*-\s*\d{6}')

# Header/footer patterns to skip
skip_pattern = re.compile(
    r'(QUEENSLAND GOVERNMENT|GAZETTE|COLUMN|Page \d+ of|Schedule 3|'
    r'WIC Table|per \$100|excluding GST|Division|Not$|applicable$|'
    r'Labour Hire$|NOTE:|WIC Table Notes|WICs and rates)'
)

rows = []
pending_prefix = None
pending_suffix_idx = None  # index of row needing a suffix line

for i, line in enumerate(all_lines):
    if not line:
        pending_prefix = None
        continue

    # Skip sub-WIC range listings under Group Training entries
    if range_pattern.match(line):
        pending_prefix = None
        continue

    # Check if this line is a suffix for the previous wrapped entry
    if pending_suffix_idx is not None:
        idx = pending_suffix_idx
        pending_suffix_idx = None
        # Only append if it looks like a plain text continuation (not a header or new data line)
        if (not full_pattern.match(line)
            and not short_pattern.match(line)
            and not re.match(r'^[A-S]\s+[A-Z]', line)
            and not re.match(r'^\d{2,3}\s+[A-Z]', line)
            and not skip_pattern.search(line)):
            rows[idx]['description'] = rows[idx]['description'] + ' ' + line
            continue

    # Try full match first: CODE DESCRIPTION RATE
    m = full_pattern.match(line)
    if m:
        wic_code = m.group(1)
        description = m.group(2)
        rate = float(m.group(3))
        pending_prefix = None
        rows.append({
            'wic_code': wic_code,
            'description': description,
            'industry_rate_pct': rate,
        })
        continue

    # Try short match: CODE RATE (wrapped description on previous line)
    m = short_pattern.match(line)
    if m:
        wic_code = m.group(1)
        rate = float(m.group(2))
        description = pending_prefix if pending_prefix else ''
        pending_prefix = None
        rows.append({
            'wic_code': wic_code,
            'description': description,
            'industry_rate_pct': rate,
        })
        # The next line might be a suffix (tail of the wrapped description)
        pending_suffix_idx = len(rows) - 1
        continue

    # Not a data line — could be a wrapped description prefix or a header/footer
    if (not skip_pattern.search(line)
        and not re.match(r'^[A-S]\s+[A-Z]', line)       # division header
        and not re.match(r'^\d{2,3}\s+[A-Z]', line)     # subdivision/group header
        and not re.match(r'^\d+$', line)                 # page number
        and len(line) > 5):
        pending_prefix = line
    else:
        pending_prefix = None

df = pd.DataFrame(rows)
print(f'{len(df)} rows parsed')
df.head()

562 rows parsed


,wic_code,description,industry_rate_pct
0,011103,Nursery Production (Under Cover),2.759
1,011204,Nursery Production (Outdoors),2.759
2,011305,Turf Growing,2.759
3,011406,Floriculture Production (Under Cover),2.759
4,011507,Floriculture Production (Outdoors),2.759


In [5]:
# Separate standard WICs from labour hire divisional WICs
is_labour_hire = df['wic_code'].str.match(r'^[A-S]\d{5}$')
is_group_training = df['wic_code'].str.startswith('8101G')

print(f'Standard WICs: {(~is_labour_hire & ~is_group_training).sum()}')
print(f'Labour hire divisional WICs: {is_labour_hire.sum()}')
print(f'Group training WICs: {is_group_training.sum()}')
print(f'Total: {len(df)}')

Standard WICs: 539
Labour hire divisional WICs: 19
Group training WICs: 4
Total: 562


In [6]:
# Extract ANZSIC class code from the WIC code
# Standard WICs: first 4 digits are ANZSIC class (e.g. 011103 -> 0111)
# Labour hire WICs: letter + 5 digits, no direct ANZSIC class
# Group training WICs: 8101G1-G4, ANZSIC class = 8101

def extract_anzsic(wic_code):
    if wic_code.startswith('8101G'):
        return '8101'
    if re.match(r'^[A-S]', wic_code):
        return None  # labour hire divisional codes don't map to a single ANZSIC class
    return wic_code[:4]

df['anzsic_class_code'] = df['wic_code'].apply(extract_anzsic)
print(f"Unique ANZSIC classes: {df['anzsic_class_code'].nunique()}")
print(f"Rows with no ANZSIC mapping (labour hire): {df['anzsic_class_code'].isna().sum()}")

Unique ANZSIC classes: 503
Rows with no ANZSIC mapping (labour hire): 19


In [7]:
# Check the 3 wrapped descriptions parsed correctly
spot_checks = ['224019', '911305', '942219']
df[df['wic_code'].isin(spot_checks)][['wic_code', 'description']]

,wic_code,description
184,224019,Sheet Metal Product Manufacturing (except Meta...
510,911305,"Sports and Physical Recreation Venues, Grounds..."
524,942219,Electronic (except Domestic Appliance) and Pre...


In [8]:
df.describe()

,industry_rate_pct
count,562.000000
mean,2.215055
std,1.640663
min,0.121000
25%,1.076250
50%,1.968000
75%,3.010750
max,18.000000


In [9]:
# Labour hire divisional WICs
df[df['wic_code'].str.match(r'^[A-S]\d{5}$')][['wic_code', 'description', 'industry_rate_pct']]

,wic_code,description,industry_rate_pct
543,A01100,"Agriculture, Forestry and Fishing",3.188
544,B06000,Mining,1.685
545,C11000,Manufacturing,3.218
546,D26000,"Electricity, Gas, Water and Waste Services",1.011
547,E30000,Construction,6.696
548,F33000,Wholesale Trade,1.673
549,G39000,Retail Trade,2.285
550,H44000,Accommodation and Food Services,2.330
551,I46000,"Transport, Postal and Warehousing",3.701
552,J54000,Information Media and Telecommunications,1.002


In [10]:
df.shape

(562, 4)

In [11]:
# Export to Parquet
df.to_parquet('QLD_WIC.parquet', index=False)
print('Saved QLD_WIC.parquet')

Saved QLD_WIC.parquet
